# WS7 — The Capstone: Conservative Offline RL Under the Full Honesty Battery

**Workstream 7 of the pitch-sequencing rigor ladder — the capstone (Phase B), and the study's closing chapter.**
This notebook builds a **conservative fitted-Q-iteration** (FQI) policy — the most flexible model in the study
(boosted-tree function approximation inside a look-one-pitch-ahead Bellman-optimality backup) — and subjects it
to the **full SPEC §9 honesty battery**. SPEC §12 names the offline-RL rung *the hardest thing in the project*;
WS7 is built to be, deliberately, its **most humble** unit: the most ambitious model, wearing the most safety
gear.

## The OPE-before-policy contract, enforced *twice*

SPEC §0.3 is absolute: *build and self-test the OPE harness (it must recover the observed policy's value)
before optimizing any policy.* WS7 enforces it with **two gates** before any policy value is reported —
**behavior-policy recovery** (the yardstick must reproduce the pitcher's own value on the held-out rows) and a
**logged-bandit fixture regression** (the estimators must recover a *known* target value on a toy problem). If
**either** fails, the run prints `FAILED_GATE` and stops — nothing downstream is interpretable.

## The chapter's deliverable is the frontier figure

WS7 does not report a coach-actionable run gain; it reports the study's **summary object**. The SPEC §10 final
figure places every policy in the study — behavior, the WS4 bandit, the WS5 MDP designs, the WS7 FQI policies
at each `α` — as a labelled point across five axes: OPE run value (with a one-sided 95% lower bound),
predictability-in-bits `B_seq`, exploitability vs equilibrium, total-variation distance from behavior, and
compute. Read as a whole, it is the trade-off the whole ladder resolves into.

## The isolation principle (the chapter's real content)

The flexible FQI *does* beat the habit-based behavior policy in raw value — but that gain is **count-driven**
(optimising the count, which value is dominated by), **not sequencing**. The WS4 lesson (`C → O` isolates
sequencing) and the WS5 lesson (trigger-count gap isolates the setup) apply, unchanged, at the top of the
ladder. So the verdict rests — a sanctioned strengthening of decision **D50** — not on the raw
`FQI-O − behavior` gap but on the **O-vs-count isolation** `(FQI-O − FQI-count)`, scored against WS4's measured
myopic ceiling `+0.003` (decision **D40**).

## The DATA_MODE toggle

This notebook is a **scaffold**. Phase 2 runs it on the real Statcast decision table; here a single toggle,
`DATA_MODE`, selects the world:

- `'synth_positive'` — the oracle's **positive world** (a planted velocity-transition whiff boost). **Default**,
  because it exercises the full capstone story end-to-end.
- `'synth_null'` — the oracle's **null world** (no ordered *outcome* effect); the verdict is `RL_NO_CLAIM`.
- `'real'` — the real decision table (RUNBOOK Steps 1, WS3, WS7.1); needs `--ws3-dir` (WS3's behavior + `q̂`),
  optionally a WS5 report for the cross-check.

## How to read this notebook

Every code step is bracketed by plain-worded markdown: **before** each cell we say what will happen and why;
**after** each cell we say how to read what came out. Numbers that depend on the real data are `{PLACEHOLDER}`
in the companion `PAPER.md`; here they simply appear when you run the cell. The **completed-validation** numbers
quoted in the markdown are *observed results* from the committed WS7a run, not aspirations. The **Results**
section (§10) is *branched* on two axes — **verdict** (`CERTIFIED` / `DIRECTIONAL` / `NO_CLAIM`–`ABSENT` /
`INCONCLUSIVE`) and **exploitability** (`E-cheap` / `E-costly`) — a code cell inspects the report and prints
which branch fired; the markdown that follows holds the pre-written interpretation for every branch. The exact
formulas live in `THEORY.md`; the exact code in `workstreams/ws7_offline_rl/model.py` and `run_ws7.py`; the
plain-English tour in `SEAN-README.md`.

> **Scale note.** WS7 is the study's **longest CPU step after WS3**: the O-view LightGBM FQI refits `× n_iter`,
> plus the value-vs-λ exhibit, plus the **FQE refit cluster bootstrap** (§5, refits FQE once per `view × α` per
> replicate). The **committed-validation** numbers quoted below come from the full run; the small in-notebook
> budgets below reproduce the **verdicts** and the qualitative story, not the exact CI widths. On real data,
> lower `--fqe-boot` to 50–100 if the refit bootstrap is slow (it changes only the CI resolution).

## 1. Setup

We put the repository root on `sys.path`, load the shared study config (the seed and the `π_α` grid come from
it), pick the world with `DATA_MODE`, and set the two FQI views and the modest in-notebook budgets. Nothing
here is WS7-specific modelling — it is the shared plumbing every workstream opens with.

In [ ]:
import sys
import json
import platform
import tempfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

# --- locate the repository root (works from repo root or from notebooks/) ---
REPO_ROOT = Path.cwd()
for _p in [Path.cwd(), *Path.cwd().parents]:
    if (_p / "pyproject.toml").exists() and (_p / "workstreams").is_dir():
        REPO_ROOT = _p
        break
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

# --- shared foundation (WS0) ---
from pitchseq.config import load_config
from pitchseq.splits import make_splits
from pitchseq.families import FAMILIES
from pitchseq.eval import ope

# --- WS7 (this workstream) ---
from workstreams.ws7_offline_rl.model import (
    FQI_VIEWS, N_COUNT, XX_INDEX, FRONTIER_COLUMNS,
    DEFAULT_LAMBDA, DEFAULT_SUPPORT_FLOOR, DEFAULT_N_ITER, DEFAULT_DISRUPTION_BETA,
    ConservativeFQI, behavior_support_penalty, conservative_greedy,
    BatterResponseModel, payoff_matrices, equilibrium_value, exploitability,
    assemble_frontier, plot_frontier, count_state,
)
# The runnable pipeline + the D40 ceiling constant (no re-implementation of the pipeline here):
from workstreams.ws7_offline_rl.run_ws7 import run_ws7, _format_headline, D40_MYOPIC_CEILING

CONFIG = load_config()
SEED = int(CONFIG.get("seeds", {}).get("global", 20260713))

# The world this run analyses: 'synth_positive' | 'synth_null' | 'real'.
# Default 'synth_positive': it plants a velocity-transition whiff boost, so the setup story (a state-value
# effect a sequential planner CAN represent but a myopic bandit cannot) is exercised end-to-end.
DATA_MODE = "synth_positive"

# The two FQI state representations (model.FQI_VIEWS): the rich ordered O view (LightGBM function approx)
# and the coarse count state (exact tabular) -- the C-analog comparison rung. Their VALUE GAP isolates
# sequencing from count-driven gain (the WS4 O-C / WS5 trigger-count lesson at the RL level).
VIEWS = list(FQI_VIEWS)                               # ("O", "count")
assert VIEWS[0] == "O"

# The pi_alpha softening grid (SPEC 9): 0 = behavior (the OPE baseline), 1 = the pure conservative target.
ALPHAS = [float(a) for a in CONFIG.get("ope", {}).get("conservative_alpha", [0.0, 0.1, 0.25, 0.5, 1.0])]
TOP_A = max(ALPHAS)

# FQI pessimism dials (model defaults): the CQL-lite behaviour-support penalty.
LAM = DEFAULT_LAMBDA            # 0.05  -- penalty coefficient (reward units)
FLOOR = DEFAULT_SUPPORT_FLOOR   # 0.02  -- behaviour-probability floor (== the SPEC 9 support_threshold)
N_ITER = DEFAULT_N_ITER         # 6     -- Bellman-optimality backups (early-stops on small drift; K=3 in validation)
BETA = DEFAULT_DISRUPTION_BETA  # 0.12  -- exploitability anticipation-disruption coefficient
LAMBDAS = sorted({0.0, LAM, 2 * LAM, 4 * LAM})       # the value-vs-lambda pessimism-exhibit grid

# --- in-notebook budgets (a quick scaffold pass; the committed run uses the full data + fqe_boot 100) ---
NB_N_GAMES = 300          # synthetic-world size (ignored when DATA_MODE == 'real')
NB_FQE_BOOT = 20          # FQE REFIT-bootstrap replicates (the resolving CI; small here for speed)
NB_N_BOOT = 40            # step-wise-DR / exploitability contribution-bootstrap replicates (CI width only)
WORLD_SEED = 7            # the seed run_ws7 uses to build a synthetic world (its default)
FIDX = {f: i for i, f in enumerate(FAMILIES)}        # family -> action index

# Phase-2 real inputs (Step 1 builds the decision table; WS3 is REQUIRED; WS5 report is optional).
REAL_TABLE_PATH = REPO_ROOT / "data" / "processed" / "decision_table.parquet"
WS3_DIR = REPO_ROOT / "results" / "ws3"
WS5_REPORT = REPO_ROOT / "results" / "ws5" / "ws5_report_real.json"

print(f"repo root : {REPO_ROOT}")
print(f"DATA_MODE : {DATA_MODE}")
print(f"views     : {VIEWS}   (O = LightGBM ordered, count = exact tabular; gap isolates sequencing)")
print(f"alphas    : {ALPHAS}   top alpha : {TOP_A}   ceiling(D40) : {D40_MYOPIC_CEILING}")
print(f"pessimism : lam={LAM}  floor={FLOOR}  n_iter={N_ITER}  beta={BETA}  lambda grid={LAMBDAS}")
print(f"budgets   : n_games={NB_N_GAMES}  fqe_boot={NB_FQE_BOOT}  n_boot={NB_N_BOOT}")
print(f"seed      : {SEED}   world_seed : {WORLD_SEED}")

### Plotting style (fixed, colorblind-safe view colours)

We use the Okabe–Ito qualitative palette (colorblind-safe), **byte-identical to WS1–WS6**, and map each FQI
view to the colour of the measurement view it mirrors — `O` (vermillion, the ordered view) and `count` (blue,
the context/count analog) — so a figure here reads the same as the ablation plots elsewhere. `REF_COLOR`
(black) is the neutral colour for reference lines (the D40 ceiling, the behavior baseline).

In [ ]:
# Okabe-Ito qualitative palette (colorblind-safe) -- identical to WS1/WS2/WS3/WS4/WS5/WS6.
OKABE_ITO = {
    "orange":         "#E69F00",
    "sky_blue":       "#56B4E9",
    "bluish_green":   "#009E73",
    "yellow":         "#F0E442",
    "blue":           "#0072B2",
    "vermillion":     "#D55E00",
    "reddish_purple": "#CC79A7",
    "black":          "#000000",
}
# Fixed view -> colour, keyed by the C / O measurement view each FQI state mirrors (identical to WS3/WS4/WS5).
VIEW_COLORS = {
    "O":     OKABE_ITO["vermillion"],    # ~ O   (full ordered current-PA history)
    "count": OKABE_ITO["blue"],          # ~ C   (context / count only)
}
BEHAVIOR_COLOR = OKABE_ITO["bluish_green"]
REF_COLOR = OKABE_ITO["black"]           # neutral: the D40 ceiling and behavior baseline reference lines

plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 110, "font.size": 11,
    "axes.titlesize": 12, "axes.titleweight": "bold",
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": False, "figure.autolayout": True,
})


def style_axes(ax):
    """Left+bottom spines only; no top/right. Returns the axis for chaining."""
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    return ax


def new_fig(figsize=(7.2, 4.2)):
    """One figure, one axis, pre-styled."""
    fig, ax = plt.subplots(figsize=figsize)
    style_axes(ax)
    return fig, ax

## 2. Inputs — the inheritance graph (what the capstone consumes from every prior rung)

WS7 is the top of the ladder, so it *consumes* the outputs of the whole study and re-implements nothing:

- **WS0 (the foundation).** The shared decision table, the temporal splits, the reward `R = −delta_run_exp`,
  the feasible-action mask, and — above all — the **OPE machinery** (`eval/ope`): DM/IPS/SNIPS/DR/step-wise
  DR/FQE, the `π_α` mixture, and behavior-policy recovery, all built and self-tested against analytic truth
  *before* any policy was optimized (SPEC §0.3). WS7 scores every value through it and re-implements none of it.
- **WS3 (required, decision D33).** The behavior propensities `μ(a | s)` for the `C` and `O` views (the OPE
  denominator and the `B_seq` predictor pair) and the `O` outcome stack `q̂(s, a)` (the step-wise-DR control
  variate and the exploitability payoff base). WS7 **loads** these on real data (`--ws3-dir`) and refits none
  of them; on a synthetic world it fits small WS3 stacks internally (as WS4/WS5 do).
- **WS4 (the myopic ceiling, decision D40).** The measured `+0.003` ceiling the O-vs-count isolation must clear
  to certify — WS4's `E[R | s, a] = v(s) + δ(s, a)` decomposition, lifted to the RL level.
- **WS5 (the refit bootstrap, decision D44; and the cross-check).** WS7 reuses WS5's paired FQE **refit
  cluster bootstrap** (`_fqe_refit_bootstrap`) and `onehot_tabular_regressor` **verbatim**, and optionally
  reads a WS5 report for the tabular-MDP cross-check.

The cell below loads the world and shows the split sizes (the FQI fits on train 2021–2023; the OPE scores the
same held-out validation + locked-test rows the rest of the study used).

In [ ]:
def load_world(mode):
    """Return (decision_table, truth_meta) for the chosen world, matching run_ws7's world build."""
    if mode == "real":
        if not REAL_TABLE_PATH.exists():
            raise FileNotFoundError(
                f"real decision table not found at {REAL_TABLE_PATH}. Build it with "
                "`python -m pitchseq.build_table` (RUNBOOK Step 1) and run WS3 (Step WS3) first."
            )
        return pd.read_parquet(REAL_TABLE_PATH, engine="pyarrow"), {"world": "real"}
    from pitchseq.decision_table import build_decision_table
    from pitchseq.synth import make_null_world, make_positive_world
    if mode == "synth_null":
        raw, truth = make_null_world(n_games=NB_N_GAMES, seed=WORLD_SEED, innings_per_game=6)
    elif mode == "synth_positive":
        raw, truth = make_positive_world(n_games=NB_N_GAMES, seed=WORLD_SEED, innings_per_game=6,
                                         effect_size=0.5, velo_gap_threshold=5.0)
    else:
        raise ValueError(f"unknown DATA_MODE {mode!r}")
    return build_decision_table(raw), truth


table, truth = load_world(DATA_MODE)
splits = make_splits(table, CONFIG)["primary"]
train = table.loc[splits["train"].to_numpy()].reset_index(drop=True)
val = table.loc[splits["val"].to_numpy()].reset_index(drop=True)
test = table.loc[splits["test"].to_numpy()].reset_index(drop=True)
eval_rows = pd.concat([val, test], ignore_index=True) if len(test) else val.copy()

print(f"world      : {truth.get('world', DATA_MODE)}")
print(f"total rows : {len(table):,}   columns: {len(table.columns)}")
print(f"train rows : {len(train):,}   (2021-2023 -- the FQI is fit here)")
print(f"eval rows  : {len(eval_rows):,}   (validation + locked test -- the rows the OPE scores)")
print(f"WS3 dir    : {WS3_DIR}   exists={WS3_DIR.exists()}   (required on real data; D33)")
print(f"WS5 report : {WS5_REPORT}   exists={WS5_REPORT.exists()}   (optional cross-check)")
if DATA_MODE == "synth_positive":
    print(f"planted    : {truth.get('mechanism')}  effect={truth.get('effect')}  "
          f"threshold={truth.get('threshold')}")

## 3. Method — conservative fitted-Q iteration (decision D48)

WS7 improves a policy by iterating the **Bellman-optimality** backup — unlike `eval/ope.fqe`, which *evaluates*
a fixed target. With `𝓕(s)` the feasible set and `pen(s, a)` the support penalty, the penalised backup
(symbol-faithful to the `model.py` docstrings) is

$$Q_{k+1}(s, a) \;\leftarrow\; r(s, a) \;+\; \mathbf{1}[s\ \text{non-terminal}]\;
  \max_{a' \in \mathcal{F}(s')}\big[\,Q_k(s', a') - \lambda\,\mathrm{pen}(s', a')\,\big],$$

applied backward over the PA steps (each iteration propagates value one pitch back). `Q_0` is the reward
regression `E[r | s, a]`; iteration stops at `n_iter` **or** when the mean taken-action drift drops below
`1e-4`. The PA horizon is short, so a handful of backups reaches the finite-horizon fixed point (**`K = 3`** in
the completed validation).

**Pessimism's role — avoid rewarding the unexplored.** The support penalty is the documented **indicator**
variant

$$\mathrm{pen}(s, a) = \mathbf{1}\!\big[\hat\mu(a \mid s) < \text{floor}\big], \qquad \text{floor} = 0.02,$$

`1` on actions the logging policy takes below the floor (their bootstrapped `Q` is off-support and
untrustworthy). `λ = 0.05` scales it in reward units, subtracted inside the backup and the greedy so the
improved policy is pushed back onto behavior support. The floor **is** the SPEC §9 out-of-support diagnostic
(a hard behavior-probability floor). This is the CQL-lite pessimism — SPEC §9's conservative discipline pushed
*inside* the Bellman backup, not only the final softening — and it is honestly an *approximation* of CQL's
regulariser, not CQL itself (`THEORY.md` §2).

**The two backends — the ladder's final C-vs-O contrast.** `ConservativeFQI` runs the *same* backup two ways:
the `count` view delegates to the exact tabular `fitted_q_iteration` (each `Q_{k+1}` an exact per-`(s,a)` group
mean), and the `O` view runs a LightGBM regressor refit per iteration, sweeping the counterfactual Q-grid by
swapping `action_family` over the 8 families (the WS3 `q_grid` pattern). Their value gap is the RL analog of
WS4's `O − C` and WS5's trigger-count gap.

**A real divergence bug, told straight.** During the build the tests caught a genuine FQI **divergence**. A row
with an *empty* feasible mask makes `max_{a′∈𝓕(s′)}[·]` a max over nothing, and the `−1e18` feasibility
sentinel `_NEG_INF` leaked into the bootstrap target `r + V(s′)`, which then diverged as it propagated backward.
The fix (`_penalized_state_value`) is a **defensive max-over-all-actions guard** on empty-feasible rows, so the
sentinel never enters a target; the pipeline additionally passes an *effective* feasible mask (empty rows → all
non-`XX`), making the guard belt-and-braces. It is recorded because it is exactly the silent-divergence failure
a fitted-Q iteration is prone to — caught by tests before it reached a number.

The cell below runs the **whole gates-first pipeline once** (`run_ws7`); every downstream exhibit reads this
one report.

In [ ]:
def run_full_pipeline(mode):
    """Run run_ws7 for the chosen world; return the report dict (one run feeds every exhibit below)."""
    tmp = tempfile.mkdtemp(prefix="ws7_nb_")
    if mode == "real":
        ws3 = str(WS3_DIR) if WS3_DIR.exists() else None
        ws5 = str(WS5_REPORT) if WS5_REPORT.exists() else None
        return run_ws7(source=str(REAL_TABLE_PATH), ws3_dir=ws3, ws5_report=ws5, synth="off", out=tmp,
                       alphas=ALPHAS, lam=LAM, floor=FLOOR, n_iter=N_ITER, lambdas=LAMBDAS, beta=BETA,
                       n_boot=NB_N_BOOT, fqe_boot=NB_FQE_BOOT, seed=WORLD_SEED, write_outputs=True)
    world = "positive" if mode == "synth_positive" else "null"
    return run_ws7(synth=world, out=tmp, n_games=NB_N_GAMES, alphas=ALPHAS, lam=LAM, floor=FLOOR,
                   n_iter=N_ITER, lambdas=LAMBDAS, beta=BETA, n_boot=NB_N_BOOT, fqe_boot=NB_FQE_BOOT,
                   seed=WORLD_SEED, write_outputs=True)


R7 = run_full_pipeline(DATA_MODE)
print(_format_headline(R7))

### Reading the FQI diagnostics (convergence + conservatism)

`fqi_diagnostics` reports, per view: the **Q-drift trace** (a decreasing tail is convergence; a *growing* trace
would flag function-approximation divergence — surfaced, not hidden), the **penalty share** among feasible
cells, and the **pessimism-bites** fraction (the share of decisions where the *unpenalised* feasible arg-max is
itself off-support — where conservatism actually changes the pick). *Completed validation:* the backup converged
in `K = 3` effective backups; the penalty bit `~2%` of decisions at the `0.02` floor. On real data with WS3's
**contextual** `μ̂` the penalty bites more (a feasible family can be off-support in a specific count), unlike the
coarse count behavior where SPEC-4 feasibility (`≥3%`) already implies support.

In [ ]:
fd = R7["fqi_diagnostics"]
fit_s = R7.get("fit_seconds", {})
print(f"{'view':<7}{'n_iter':>7}{'final_drift':>13}{'converged':>11}{'penalty_share':>15}"
      f"{'pessimism_bites':>17}{'fit(s)':>9}")
for v in R7["views"]:
    d = fd.get(v, {})
    print(f"{v:<7}{d.get('n_iter_run'):>7}{d.get('final_drift', float('nan')):>13.6f}"
          f"{str(d.get('converged')):>11}{d.get('penalty_share', float('nan')):>15.3f}"
          f"{d.get('pessimism_bites_frac', float('nan')):>17.1%}{fit_s.get(v, float('nan')):>9.1f}")
print("\ndrift trace (O):", [round(x, 5) for x in fd.get("O", {}).get("drift_per_iter", [])])

### The pessimism exhibit — "conservatism is free honesty until it isn't"

Refitting FQI-`O` at each `λ` in the grid (softened at the top `α`) and reading the held-out FQE value, the mean
TV-from-behavior, and how often the penalty bites: as `λ` rises the policy retreats toward behavior support —
deviation and exploitability fall — until the penalty over-constrains it. *Completed validation:* the value is
`+0.0328` at `λ = 0` and `+0.0354` at `λ ≥ 0.05` — a modest penalty **slightly raises** the held-out value (it
pulls the flexible model off the over-rated off-support actions it would otherwise chase) while lowering
deviation. Free honesty, here even slightly value-positive.

In [ ]:
curve = R7.get("pessimism", {}).get("curve", [])
print(f"value-vs-lambda (FQI-O @ alpha={R7['pessimism'].get('top_alpha')}):")
print(f"  {'lambda':>7}{'FQE value':>12}{'mean TV':>10}{'penalty_share':>15}{'bites':>9}")
for row in curve:
    print(f"  {row['lambda']:>7.2f}{row['fqe_value']:>12.4f}{row['mean_tv']:>10.3f}"
          f"{row['penalty_share']:>15.3f}{row['pessimism_bites_frac']:>9.1%}")

fig, ax = new_fig(figsize=(6.6, 4.0))
lam_x = [r["lambda"] for r in curve]
ax.plot(lam_x, [r["fqe_value"] for r in curve], "-o", color=VIEW_COLORS["O"], label="FQI-O FQE value")
ax.plot(lam_x, [r["mean_tv"] for r in curve], "-s", color=BEHAVIOR_COLOR, label="mean TV from behavior")
ax.set_xlabel(r"pessimism coefficient $\lambda$")
ax.set_ylabel("value / deviation")
ax.set_title("pessimism exhibit: conservatism is free honesty until it isn't")
ax.legend(frameon=False, fontsize=9)
plt.show()

## 4. The two gates — OPE-before-policy, enforced twice (SPEC §0.3)

Before any policy value is computed the pipeline runs **two** gates; either failing prints `FAILED_GATE` and
stops (`THEORY.md` §5 derives what each catches):

- **Gate 1 — behavior-policy recovery** *(calibrates the yardstick).* Set the target `= μ` on the held-out
  logged rows; every estimator must return the observed value with IPS weights ≡ 1. Catches a miscalibrated
  yardstick (a propensity/reward join error). *Completed validation:* observed value `+0.0197` (null),
  `+0.0345` (positive), IPS weights ≡ 1.
- **Gate 2 — the logged-bandit fixture regression** *(checks the estimator implementations).* On a bandit with
  a *known* target value, OPE must recover both the logging value and the known target within CI. Catches an
  estimator bug real data cannot expose. *Completed validation:* known `+0.6148` vs estimated `+0.6229`, within
  CI.

> **`FAILED_GATE` = stop.** If either gate fails, nothing downstream is interpretable — OPE that cannot recover
> a *known* value cannot be trusted to value a policy. The pipeline stops before the FQI is even fit.

In [ ]:
rec = R7["behavior_recovery"]
bg = R7["bandit_gate"]
print(f"GATE STATUS : {R7['gate']}")
print(f"  Gate 1 (behavior-policy recovery): passed={rec['passed']}  observed_value={rec['observed_mean']:+.4f}"
      f"  IPS weights unit={rec['ips_weights_unit']}  tol={rec.get('tol')}")
print(f"  Gate 2 (logged-bandit regression): passed={bg['passed']}  recovery_passed={bg['recovery_passed']}")
print(f"           known target={bg['target_known']:+.4f}  estimated={bg['target_estimate']:+.4f}"
      f"  within_CI={bg['target_ok']}")
if R7["gate"] != "PASSED":
    print("\n*** FAILED_GATE: the run stopped before any policy value. Nothing below is interpretable. ***")

## 5. The §9 battery per policy × α — refit-FQE (the D44 inheritance from WS5)

Each softened target `π_α` (per `view × α`) is scored on the held-out rows by **step-wise DR** (with WS3's `q̂`
control variate; used *directionally* — its per-PA weight product is heavy-tailed) and a **refit-bootstrap
FQE** (the resolving instrument), plus the SPEC §9 diagnostics (ESS, TV).

**Why a *refit* bootstrap (decision D44, recap).** FQE's per-episode contribution is the initial-state value
`V(s_0)`, and **every PA starts in the same state**, so a resampling bootstrap of those contributions is
structurally degenerate — its CI collapses to a point (the WS5 finding, `../ws5_tabular_mdp/THEORY.md` §7). WS7
therefore reuses WS5's `_fqe_refit_bootstrap` **verbatim**: resample pitcher-game clusters of episodes, refit
FQE from scratch per `(view, α)` — the *same* resample across arms, so the gaps are **paired** — and read
percentile CIs off the replicates. Degenerate CIs print `n/a`, never a fake interval.

*Completed validation (positive world):* refit-FQE FQI-O@0.5 `+0.0340` `[+0.0292, +0.0423]`, FQI-O@1 `+0.0354`
`[+0.0298, +0.0464]`, FQI-count@1 `+0.0357` `[+0.0264, +0.0520]`; ESS decays `100% → ~22%` as `α` rises (the
price of deviating from behavior).

In [ ]:
print(f"{'view':<7}{'alpha':>6}{'FQE':>10}{'FQE CI (refit)':>24}{'lo95':>9}{'stepDR':>10}{'ESS%':>8}{'TV':>8}")
for v in R7["views"]:
    for a in R7["alphas"]:
        c = R7["ladder"][v][a]
        fq = c["FQE"]
        ci = "n/a" if fq.get("degenerate") else f"[{fq['ci95'][0]:+.4f},{fq['ci95'][1]:+.4f}]"
        print(f"{v:<7}{a:>6.2f}{fq['value']:>10.4f}{ci:>24}{fq['lower_95']:>9.4f}"
              f"{c['stepwise_DR']['value']:>10.4f}{c['ess_frac']:>8.1%}{c['mean_tv']:>8.3f}")
fr = R7.get("fqe_refit", {})
print(f"\nrefit bootstrap: {fr.get('n_boot')} replicates, {fr.get('n_fits')} FQE refits, "
      f"{fr.get('seconds', float('nan')):.1f}s (committed run: 100 reps on real data).")

In [ ]:
# ESS decay: the price of deviating from behavior, per view.
fig, ax = new_fig(figsize=(6.6, 4.0))
for v in R7["views"]:
    ess = [R7["ladder"][v][a]["ess_frac"] for a in sorted(R7["alphas"])]
    ax.plot(sorted(R7["alphas"]), ess, "-o", color=VIEW_COLORS[v], label=f"{v}")
ax.axhline(1.0, color=REF_COLOR, lw=0.8, ls=":")
ax.set_xlabel(r"softening $\alpha$  (0 = behavior, 1 = pure conservative target)")
ax.set_ylabel("effective sample size fraction")
ax.set_title("ESS decay as the policy deviates from behavior")
ax.legend(frameon=False, title="FQI view")
plt.show()

## 6. The isolation principle — the D50 strengthening (the chapter's real content)

It would be **easy** to headline the raw win: FQI-O beats the habit-based behavior policy by a large,
CI-clear-of-zero margin. **But that gain is count-driven** — the flexible policy optimises the *count* (which
value is dominated by; Tango et al., 2007), which has nothing to do with sequencing. The WS4 lesson (`C → O`
isolates sequencing from count-driven gain) applies unchanged at the top of the ladder.

So the verdict rests — as a sanctioned strengthening of D50's literal wording — not on the raw
`FQI-O − behavior` gap but on the **O-vs-count isolation** `(FQI-O − FQI-count)`, the RL analog of WS4's
`O − C` and WS5's trigger-count gap, scored against the `+0.003` myopic ceiling:

$$\text{raw} = \underbrace{(V_{\text{FQI-count}} - V_{\text{behavior}})}_{\text{count-driven}} + \underbrace{(V_{\text{FQI-O}} - V_{\text{FQI-count}})}_{\text{sequencing isolation}}.$$

*Completed validation:* the raw gap is `+0.035` (CI clears 0) — but the **isolation is `−0.0003`** (CI
`[−0.0129, +0.0157]`, **within OPE noise**), while the isolation robustly **discriminates the worlds**
(positive `−0.0003 >` null `−0.0016`) at every scale probed (160/300/400/600 games).

In [ ]:
top = TOP_A
g = R7["gaps"][top]
gb, gc = g["o_minus_behavior"], g.get("o_minus_count")
def _ci(e):
    return "n/a" if e.get("degenerate") else f"[{e['ci95'][0]:+.4f},{e['ci95'][1]:+.4f}]"
print(f"GAPS at alpha={top:.2f}  (paired refit-FQE; ceiling=+{R7['ceiling']:.3f}):\n")
print(f"  RAW  FQI-O - behavior   (count-inclusive; NOT the basis):")
print(f"       FQE={gb['FQE']['value']:+.4f} {_ci(gb['FQE'])} lo95={gb['FQE']['lower_95']:+.4f}"
      f"  | stepDR={gb['stepwise_DR']['value']:+.4f}")
if gc:
    print(f"  ISO  FQI-O - FQI-count  (the SEQUENCING ISOLATION -- the D50 basis):")
    print(f"       FQE={gc['FQE']['value']:+.4f} {_ci(gc['FQE'])} lo95={gc['FQE']['lower_95']:+.4f}"
          f"  | stepDR={gc['stepwise_DR']['value']:+.4f}")
    clears = (not gc["FQE"].get("degenerate")) and np.isfinite(gc["FQE"]["lower_95"]) \
        and gc["FQE"]["lower_95"] > R7["ceiling"]
    print(f"       isolation lower-95 clears +{R7['ceiling']:.3f} ceiling: {clears}")
print(f"\n  D24 sequential agreement (stepDR vs FQE on FQI-O@{top:.2f}): {R7['seq_agreement'].get('verdict')}")
print(f"  the raw gain, if any, is COUNT-DRIVEN -- the isolation is what the verdict reads.")

**How easy it would have been to claim the raw number — and why we don't.** A `+0.035` improvement over
behavior, with a lower bound clear of zero, is a headline any reader would accept as "the model finds a better
policy." It *is* a better policy — but almost entirely for **count** reasons a bandit already captures, so
selling it as *sequencing* would be the exact over-claim the whole ladder was built to prevent. The isolation
subtracts the count rung's value and leaves only the sequencing-specific slice, which is tiny and, at this
scale, buried in noise. That we can *discriminate* the worlds (the positive slice reliably exceeds the null
slice) is the honest good news; that we cannot *certify* the slice is the honest bad news. Reporting both, and
resting the verdict on the isolation, is the discipline (`THEORY.md` §4).

## 7. The WS5 cross-check — corroboration only

WS7's O-vs-count isolation and WS5's tabular trigger-count gap measure the same sequencing signal with
different machinery (a flexible LightGBM FQI vs a low-variance tabular MDP). Their **directional agreement** is
read as corroboration. It is **required for `CERTIFIED`** but **not for `DIRECTIONAL`** (which rests on WS7's
own estimators), and it is honestly **noisy at small synthetic scale** — its sign is not even stable at fixture
size. *Completed validation:* WS5 trigger-count FQE gap `−0.0213` (WS5 verdict `SETUP_INCONCLUSIVE`) on the
positive world, `+0.0137` (`SEQ_NEUTRAL_MDP`) on the null — read it as directional corroboration, not a second
gate. On real data, at `~40×` the scale, the two isolation signals should agree in sign.

In [ ]:
wx = R7.get("ws5_crosscheck", {})
if wx.get("available"):
    print(f"WS5 cross-check ({wx.get('source')}):")
    print(f"  trigger-count FQE gap : {wx.get('trigger_count_fqe_gap'):+.4f}")
    print(f"  WS5 verdict           : {wx.get('ws5_verdict')}")
    print(f"  directional-positive  : {wx.get('directional_positive')}   (corroboration only; required for")
    print(f"                          CERTIFIED, not DIRECTIONAL; noisy at small synthetic scale)")
else:
    print(f"WS5 cross-check unavailable: {wx.get('reason')}  (pass --ws5-report on real data)")

## 8. Exploitability — the §10 game-theoretic capstone

The exploitability read-out is a **light game-theoretic** capstone, model-dependent by necessity (SPEC §10).

**The batter-response model (`BatterResponseModel`) — exactly what it is and isn't.** A *fixed, interpretable*
tabular `P(swing | count, family, location-bucket)`, fit once on train as beta-smoothed empirical swing rates.
The location bucket is an **execution** quantity, so it is used only to *fit* the model and is **marginalised
away at prediction** — a pitch's location is not known pre-decision, so the game uses the raw `(count, family)`
swing rate (`swing_prob`). It is **not** an adaptive batter, **not** a learned best-responder, and **not**
causal; it is one transparent opponent model.

**The per-count game (`payoff_by_count` → `payoff_matrices`).** Each of the 12 counts is a zero-sum 8×8 game
between the pitcher (row, maximiser) and a family-anticipating batter (column, minimiser):

$$R(s, a, k) = \hat q(s, a) \;-\; \beta\,p_{\text{swing}}(s, a)\,\mathbf{1}[a = k], \qquad \beta = 0.12,$$

`q̂` the common outcome model's `E[R | s, a]` (WS3's grid), the anticipation penalty applying only when the
batter guessed the thrown family. **Equilibrium via LP (`equilibrium_value`).** The pitcher's maximin is solved
as a linear program (`scipy.linprog`); by von Neumann's minimax theorem the optimal value is the game value. A
policy's **exploitability** is `V_eq − min_k (π^T R)_k ≥ 0` — its shortfall from equilibrium against a
best-responding batter (`exploitability`), scored in each row's own count game.

*Completed validation:* behavior `+0.0137`, FQI-O@1 `+0.0459`, FQI-count@1 `+0.0475` — the concentrated
conservative policies cost **`~3×` the exploitability of diffuse behavior**. `B_seq = +0.0099` bits.

In [ ]:
ex = R7["exploitability"]
tbl = ex["table"]
print(f"exploitability vs equilibrium (mean; eq value={ex['eq_value_mean']:+.4f}):")
def _exci(e):
    lo, hi = e.get("ci95", [float('nan')] * 2)
    return "" if not np.isfinite(lo) else f"  CI[{lo:+.4f},{hi:+.4f}]"
for key in ["behavior", f"O@{TOP_A:g}", f"count@{TOP_A:g}"]:
    if key in tbl:
        e = tbl[key]
        mult = e["mean"] / tbl["behavior"]["mean"] if tbl["behavior"]["mean"] else float("nan")
        print(f"  {key:<14} exploitability={e['mean']:+.4f}{_exci(e)}   ({mult:.1f}x behavior)")

bs = R7["b_seq"]
print(f"\nB_seq predictability (bits): overall={bs['b_seq_overall']:+.4f}  "
      f"seq-eligible={bs['b_seq_seq_eligible']:+.4f}")
print("by pitch-number (depth):", [(r.get('pitch_number'), round(r.get('mean_bits', 0), 4)) for r in bs['by_depth'][:6]])

In [ ]:
# The exploitability cost, one bar per policy (behavior + the top-alpha FQI policies).
keys = ["behavior", f"O@{TOP_A:g}", f"count@{TOP_A:g}"]
labels = ["behavior\n(diffuse)", f"FQI-O @ {TOP_A:g}\n(ordered)", f"FQI-count @ {TOP_A:g}\n(count)"]
present = [(k, lab) for k, lab in zip(keys, labels) if k in tbl]
vals = [tbl[k]["mean"] for k, _ in present]
cols = [BEHAVIOR_COLOR, VIEW_COLORS["O"], VIEW_COLORS["count"]][:len(vals)]
fig, ax = new_fig(figsize=(6.4, 4.0))
ax.bar(range(len(vals)), vals, color=cols, edgecolor="k", linewidth=0.5)
ax.axhline(tbl["behavior"]["mean"], color=REF_COLOR, lw=0.8, ls=":", label="behavior baseline")
ax.set_xticks(range(len(vals)))
ax.set_xticklabels([lab for _, lab in present], fontsize=9)
ax.set_ylabel("average exploitability vs equilibrium")
ax.set_title("value is bought with predictability: concentrated policies are more exploitable")
plt.show()

**The study's game-theoretic bottom line: prescriptive value and predictability trade off.** The
concentrated conservative policies find higher OPE value, but a fixed batter who *guessed along* would claw
back roughly three times as much against them as against real pitchers' diffuse mixing. There is no free
prescription: the value the model finds is bought, in part, with predictability. This is model-dependent by
necessity (SPEC §10 — the exact `~3×` moves with the batter model), but the *direction* (concentration costs
exploitability) is robust. `B_seq` — the extra bits the ordered history gives away about the *next pitch* — is
small (`+0.0099`) and rises with PA depth and count leverage; forecastable ordering is part of the same
predictability picture, though not by itself proof it helps the batter.

## 9. The frontier — the study's final deliverable (SPEC §10)

The frontier is the chapter's deliverable and the study's summary object. `assemble_frontier` builds one row per
policy — behavior, the WS7 FQI policies at each `view × α`, and the WS5 trigger design from the cross-check —
with `FRONTIER_COLUMNS = [policy_id, ope_value, ope_lower95, b_seq_bits, exploitability, tv_from_behavior,
params, wall_clock_s]`; `plot_frontier` renders the two-panel figure. *Completed validation:* **10 policy
rows** (behavior, FQI-O and FQI-count at `α ∈ {0.1, 0.25, 0.5, 1}`, and `ws5_trigger`).

In [ ]:
frontier_df = assemble_frontier(R7["frontier"])
cols = ["policy_id", "ope_value", "ope_lower95", "b_seq_bits", "exploitability", "tv_from_behavior", "params"]
print("THE FRONTIER (the study's final deliverable):\n")
print(frontier_df[cols].to_string(index=False, float_format=lambda x: f"{x:+.4f}"))

# Render the actual study figure (the two-panel plot_frontier) inline.
png = plot_frontier(frontier_df, Path(tempfile.mkdtemp(prefix="ws7_fr_")) / "frontier.png",
                    title=f"WS7 study frontier ({R7['world']})")
fig, ax = plt.subplots(figsize=(13, 5.6))
ax.imshow(plt.imread(png)); ax.axis("off")
plt.show()

**How to read it (value × bits × exploitability × TV × compute).** *Left panel:* OPE run value (y, with
the one-sided 95% lower bound as a down-whisker) vs total-variation distance-from-behavior (x), coloured by
exploitability — the prescriptive trade-off, value bought with deviation and predictability. *Right panel:*
exploitability (y) vs predictability-in-bits `B_seq` (x), point size = compute (`params`) — the game-theoretic
axis. Every policy is a labelled point in both, so all five SPEC §10 axes are legible in one figure. **No single
"winner" is crowned** — the frontier is a Pareto surface, not a leaderboard (`THEORY.md` §7): the "best" policy
depends on how a reader weights value against predictability, and the study refuses to fabricate that weighting.
**Phase 2** adds the real-data rows (`{PLACEHOLDER}`) alongside the WS4 bandit and WS5 designs, at `~40×` the
scale — the study's closing figure.

## 10. Results — branched interpretation (Verdict × Exploitability)

The result is read on two axes: the **verdict** (WS7's headline) and the **exploitability** (SPEC §10's
game-theoretic qualifier). The cell below inspects the computed report and prints which branch fired on each;
the markdown that follows holds the pre-written interpretation for every branch, and stands alone once the real
numbers arrive.

In [ ]:
top = TOP_A
v = R7["verdict"]
verdict = v.get("verdict")

# --- Verdict axis (from the pipeline's own D50 machinery) ---
seq_lo = v.get("seq_gap_lower95", float("nan"))
clears = bool(v.get("ceiling_cleared"))

# --- Exploitability axis: E-cheap if the top-alpha O policy is <= behavior exploitability, else E-costly ---
tbl = R7["exploitability"]["table"]
beh_ex = tbl["behavior"]["mean"]
o_ex = tbl.get(f"O@{top:g}", {}).get("mean", float("nan"))
e_branch = "E-cheap" if np.isfinite(o_ex) and o_ex <= beh_ex + 1e-9 else "E-costly"

print("=" * 72)
print(" WS7 RESULTS -- branch selector")
print("=" * 72)
print(f" world / mode         : {R7.get('world')}")
print(f" RAW  FQI-O-behavior  : {v.get('raw_gap'):+.4f}  (count-inclusive; NOT the basis)")
print(f" ISO  FQI-O-FQI-count : {v.get('seq_gap'):+.4f}  lo95={seq_lo:+.4f}  (ceiling +{R7['ceiling']:.3f}, "
      f"cleared={clears})")
print(f" stepDR isolation gap : {v.get('seq_stepdr_gap'):+.4f}   D24 agreement: {v.get('seq_agreement')}")
if beh_ex:
    print(f" exploitability       : behavior={beh_ex:+.4f}  FQI-O@{top:g}={o_ex:+.4f}  ({o_ex / beh_ex:.1f}x)")
print("-" * 72)
print(f" VERDICT axis         : {verdict}")
print(f" EXPLOITABILITY axis  : {e_branch}   (E-costly = value bought with predictability -- the verified pattern)")
print("=" * 72)
print(" -> read the matching branch write-ups in the markdown below.")

> **Binding reading rule (stated before any branch).** *The isolation is the basis, never the raw gain, and
> the error bar is the claim.* A sequencing claim requires the **O-vs-count isolation** (not the count-inclusive
> `FQI-O − behavior` gap) to clear `+0.003` with a **non-degenerate** refit-FQE lower bound. A raw improvement
> over behavior is *count-driven* until the isolation says otherwise; a degenerate CI is `n/a`, never a
> certification; a D24 estimator disagreement is `RL_INCONCLUSIVE`, never "it works" (decisions D50, D24).

### Verdict axis — `RL_EVIDENCE_CERTIFIED` (the strong claim)

The O-vs-count isolation's refit-FQE lower-95 clears the `+0.003` ceiling, its step-wise DR agrees in sign,
**and** the WS5 cross-check is directionally consistent: the flexible sequential policy cashes ordered-history
value on held-out data **beyond the count-driven gain and beyond the myopic ceiling** — the ladder's motivating
contrast delivered at its top rung. Before believing it, run the checklist: **support** (ESS not collapsed, oos
small, max weight moderate), **per-slice** concentration (the gap should live in `long_pa` / `two_strike`),
**λ-sensitivity** (the isolation should survive a modest penalty), and the **WS5 corroboration**. *Consequence
for the synthesis:* the arc's "real but uncertifiable" resolves to "real and certified at scale" — the Phase-2
question answered yes. Proven reachable by `test_verdict_certified_path_reachable`, so it is a real code path,
not decoration.

### Verdict axis — `RL_EVIDENCE_DIRECTIONAL` (the synth-validated pattern; the most anticipated cell)

The isolation is directional (or, on the known-positive synthetic world, world-gated: the raw improvement is
real, the isolation *discriminates* the worlds, and WS5 corroborates) but its refit-FQE CI cannot clear the
ceiling — **evidence real, statistics insufficient**. This is exactly WS4's `SEQ_INCONCLUSIVE_MYOPIC` and WS5's
`SETUP_INCONCLUSIVE`, now at the RL level. State the `n`-to-certify back-of-envelope: the isolation half-width
scales as `~1/√(n_clusters)`; WS7 is *more* variance-bound than WS5 (flexibility costs variance), so its
required multiple (`~16×`) exceeds WS5's (`~6×`), and the full data is `~40×` the fixture — comfortably past it
if the signal holds (`THEORY.md` §8). *Consequence for the synthesis:* the arc closes on "sequential value is
real and representable; certifying it is a statistics-of-scale problem," with the flexible model more
variance-bound than the tabular one.

### Verdict axis — `RL_NO_CLAIM` / `RL_EVIDENCE_ABSENT` (the honest nulls)

On the **null** world (planted-absent), or a real table with no directional isolation, the O-vs-count isolation
is `~0`/negative and WS7 claims nothing beyond the count-driven gain. `RL_NO_CLAIM` is the null world's verdict
by construction — the headline prints the count-driven explanation: any raw gain over the habit-based behavior
policy is optimising the count, not sequencing. `RL_EVIDENCE_ABSENT` is its real-data analog. Read it as "no
prescriptive sequencing edge beyond the myopic-count policy at this scale," **never as "order hurts"** — the raw
gain over behavior, if any, is count-driven and is not retold as sequencing. *Consequence for the synthesis:*
the ladder's honest terminal — the prescriptive edge is not present, or not resolvable, and the study reports
that plainly.

### Verdict axis — `RL_INCONCLUSIVE` (estimator disagreement, SPEC §9 verbatim)

The D24 sequential estimators (step-wise DR vs FQE) differ by more than the wider of their CI half-widths. Per
SPEC §9's closing rule:

> "If estimators disagree materially, the verdict is `INCONCLUSIVE`, not 'it works.'"

The value is not a coherent read at this `α`; report it as inconclusive, lean on the lower-`α` / higher-ESS
rows, and diagnose the propensity / `q̂` / support before any reading. *Consequence for the synthesis:* the
capstone's value estimate is not coherent at this scale; the frontier's FQI rows carry the caveat.

### Exploitability axis — `E-cheap` vs `E-costly`

**`E-cheap` — value gained without exploitability cost.** The FQI policy's exploitability is at or below
behavior's while its OPE value is above — better *and* no more predictable. The happy case, and a genuinely
strong prescriptive result; it is **not** what the synthetic validation shows, so treat an `E-cheap` real-data
reading with the same support checklist as `CERTIFIED`.

**`E-costly` — value bought with predictability (the verified pattern).** The FQI policy's exploitability is
above behavior's (the synthetic worlds show `~3×`), because a concentrated policy lets the fixed batter-response
model sit on it. The cross-reading with the batter's-side story is the study's game-theoretic bottom line: an
optimised pitcher who always throws the model's pick is more forecastable, and a batter who guesses along claws
back a measurable share — model-dependent by necessity, but the *direction* is robust. *Consequence for the
synthesis:* the frontier is a genuine multi-objective trade-off, not a single-number leaderboard — the "best"
policy depends on how a user weights value against predictability.

**Reading the grid.** The honest headline is a pair `(verdict, exploitability)`. The *most anticipated* real-
data cell is **`RL_EVIDENCE_DIRECTIONAL` × `E-costly`** — a real, world-discriminating sequencing signal below
the OPE certification floor, bought at a predictability cost. The *strongest* is **`RL_EVIDENCE_CERTIFIED` ×
`E-cheap`**. The *null* is **`RL_NO_CLAIM` / `ABSENT`**. The *stop* is **`RL_INCONCLUSIVE`**.

## 11. Discussion & limitations

**The capstone-arc finding (WS4 → WS5 → WS7).** Each prescriptive rung's honest negative is the next rung's
motivating contrast, and together they make one coherent finding. **WS4's greedy bandit was structurally
blind** to setups (a one-step value cannot represent an action whose payoff is a future state) and *measured*
the ceiling (`~0.003` of `~0.032`, D40). **WS5's tabular MDP proved the setup representable** (a constructed-
world test cashes it) and directionally recovered it, but bound its *certification* to the sequential-OPE
variance (D44). **WS7's flexible FQI discriminates the worlds robustly** yet is **more variance-bound than the
tabular MDP at equal scale**: its LightGBM function approximator buys representational reach (no hand-designed
trigger flag) but carries more estimation variance into the OPE, so its isolation CI is *wider* than WS5's and
it cannot cash the small setup credit the tabular design recovered. The consistent conclusion across all three:
**sequential value exists and is representable; certifying it is a statistics-of-scale problem** — the study's
Phase-2 question, stated precisely.

**Flexibility costs variance — a general finding.** Moving up the model-complexity ladder (bandit → tabular MDP
→ flexible FQI) monotonically increases representational reach *and* finite-sample OPE variance. The paradox
that the *fanciest* model is the *least* certain at a fixed data scale is not a bug — it is the bias-variance
trade-off on the OPE side of the ledger, and it is why the right model for a *certification* at moderate scale
is often the *simplest one that can represent the effect*; the flexible model earns its place at full scale.

**The frontier as the study's summary object.** WS7's deliverable is not a run gain but the frontier figure.
The whole ladder resolves into one multi-objective picture — what OPE value each policy buys, at what
predictability, exploitability, deviation, and compute — with no scalarisation imposed. The honest bottom line
is not "throw the slider" but "here is the trade-off, and here is how far each rung of rigor gets you before the
error bars close in."

**Limitations.**
1. **Exploitability is model-dependent by necessity (SPEC §10).** The batter-response model is one fixed,
   interpretable opponent (a beta-smoothed `P(swing | count, family)` marginalised over location) — not
   adaptive, not a learned best-responder, not causal. The *levels* move under a different model; the robust
   claim is the *direction* (concentration costs exploitability), not the exact `~3×`. SPEC §10 asks for
   robust/worst-case reporting, and that is the direction.
2. **The penalty form.** The support penalty is the indicator `1[μ̂ < 0.02]` — a CQL-lite *approximation*, not
   the CQL regulariser; the smooth count-based alternative is documented but not used. Floor and `λ` are config
   choices, reported openly through the value-vs-λ exhibit.
3. **`γ = 1` and the finite horizon.** The undiscounted PA return makes `γ = 1` correct, not convenient
   (`../ws5_tabular_mdp/THEORY.md` §1); the short horizon is why `K = 3` backups suffice. Long/non-terminating
   episodes would need discounting.
4. **Synthetic-scale OPE floors bind certification, not representability.** The `~0.003` ceiling is swallowed by
   the isolation CI at synthetic scale; whether the real-data isolation resolves above it is the Phase-2
   question, and the `n`-to-certify arithmetic is a back-of-envelope, not a guarantee.
5. **Deferred variants (SPEC §12).** Full CQL and model-based RL are deferred to Phase 2 (the highest model-
   error risk); WS7 ships the CPU-practical conservative FQI. A full stochastic-game solver for the
   exploitability capstone is likewise deferred — this is the light game-theoretic version.
6. **Family granularity, the `pitch_type` proxy, and one reward metric** — inherited from WS3 (SPEC §1, §4): a
   mechanism below the family level or a mislabeled pitch is only partially represented, and a different reward
   than `−delta_run_exp` would move every number.
7. **The firewall.** WS7 tests whether *acting* on the FQI value beats behavior *within support*; it does not
   certify the ordered state as *causal*. `CERTIFIED` is evidence of a better *evaluable* policy, not proof that
   *changing* the sequence *causes* the gain (`../ws3_gbdt_stack/THEORY.md` §8).

## 12. Reproducibility appendix

**Run it (RUNBOOK Step WS7).** WS7.1 (gates + FQI + the §9 battery) is the heavy step; WS7.2 (exploitability +
`B_seq` + the frontier) is folded into the same run. WS7 **requires** WS3 (Step WS3) and *optionally* a WS5
report.

```powershell
conda activate statcast; cd ~\pitch-sequencing-research
python workstreams/ws7_offline_rl/run_ws7.py --table data/processed/decision_table.parquet --ws3-dir results/ws3/ --ws5-report results/ws5/ws5_report_real.json --out results/ws7/ --fqe-boot 100
```

Synthetic self-tests (no real data, no `--ws3-dir`): `--synth null --out results/ws7_null/ --fqe-boot 40` and
`--synth positive --out results/ws7_pos/ --fqe-boot 40`.

**Outputs (`results/ws7/`).** `policy_<world>_<view>.parquet` (standard-schema `policy_prob` per view; view→
state_view O→O, count→C); `fqi_<world>_<view>.joblib` + `response_<world>.joblib` (the fitted FQI Q-models and
batter-response model; loader `load_ws7_artifacts`); `frontier_<world>.csv` + **`frontier_<world>.png`** (the
study's final figure); `ws7_report_<world>.json` (the full report — gates, ladder, gaps, WS5 cross-check,
exploitability, `B_seq`, pessimism curve, verdict); `ws7_<world>.runmeta.json` (timing / peak RAM).

**Cost.** WS7 is the **longest CPU step after WS3** — the O-view LightGBM FQI refits `× n_iter`, the value-vs-λ
exhibit adds a few more O-view refits, and the FQE **refit bootstrap** is `fqe_boot × 2 views × |α|` tabular
refits. Budget order 1–3 h on the full data; lower `--fqe-boot` to 50–100 if slow (it changes only CI
resolution). Degenerate CIs print `n/a (constant contributions)`, never a fake interval.

**Paste back, per RUNBOOK:** the entire printed **headline block** and the frontier CSV
(`Get-Content results/ws7/frontier_real.csv`).

In [ ]:
import scipy, sklearn, lightgbm
print("python      :", platform.python_version())
print("numpy       :", np.__version__)
print("pandas      :", pd.__version__)
print("scipy       :", scipy.__version__)
print("scikit-learn:", sklearn.__version__)
print("lightgbm    :", lightgbm.__version__)
print("matplotlib  :", matplotlib.__version__)
print("seed        :", SEED, " world_seed:", WORLD_SEED)
print("data mode   :", DATA_MODE)
print("views       :", VIEWS, " (O=LightGBM ordered, count=exact tabular)")
print("alphas      :", ALPHAS, " top:", TOP_A, " ceiling(D40):", D40_MYOPIC_CEILING)
print("pessimism   : lam", LAM, " floor", FLOOR, " n_iter", N_ITER, " beta", BETA)
print("budgets     : n_games", NB_N_GAMES, " fqe_boot", NB_FQE_BOOT, " n_boot", NB_N_BOOT)